### Bio Facts

In [ ]:
import random
import numpy as np
import torch
import os
from dotenv import load_dotenv

import gradio as gr
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.auto import tqdm

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DEVICE = 'cuda' if torch.cuda.is_available() else 'auto'
print(f'Using device: {DEVICE}')

In [ ]:
load_dotenv(override=True)
if not os.environ.get("HF_TOKEN"):
    raise ValueError("HF_TOKEN is not set")

In [ ]:


training_data_set = load_dataset("json", data_files="johngorithm.jsonl")
validation_data_set = load_dataset("json", data_files="validation_data.jsonl")
test_data_set = load_dataset("json", data_files="test_data.jsonl")


# Dataset sample
sample = {
    "prompt": "Who is  Johngorithm ?",
    "completion": "Johngorithm  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and leadership."
}


In [ ]:
training_data = training_data_set['train']
validation_data = validation_data_set['train']
test_data = test_data_set['train']


In [ ]:
training_data[0]

### Data preparation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(data_point):
    prompts = data_point["prompt"]
    completions = data_point["completion"]
    text = f"{prompts}\n{completions}"

    return tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
    )



In [ ]:
preprocess(sample)

In [ ]:
training_data = training_data.map(preprocess)
validation_data = validation_data.map(preprocess)
test_data = test_data.map(preprocess)

In [ ]:
training_data.set_format('torch', columns=['input_ids', 'attention_mask'])
validation_data.set_format('torch', columns=['input_ids', 'attention_mask'])
test_data.set_format('torch', columns=['input_ids', 'attention_mask'])


print('Data tokenized and ready for training!')

### Fine-Tunning

In [ ]:
# Load pre-trained model with regression head (num_labels=1 for regression)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression"
)

model.to(DEVICE)
print(f'Model loaded on {DEVICE}')

In [ ]:
training_args = TrainingArguments(
    output_dir='./checkpoints',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    report_to='none',
    seed=42
)

In [ ]:
# Define metrics for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.flatten()
    
    mae = mean_absolute_error(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    r2 = r2_score(labels, predictions)
    
    return {
        'mae': mae,
        'rmse': rmse,
        'r2': r2
    }

In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_data,
    eval_dataset=validation_data,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


In [ ]:
trainer.train()
print('Training complete!')

In [ ]:
model.save_pretrained('./fine_tuned')
tokenizer.save_pretrained('./fine_tuned')

print('Model and tokenizer saved to ./fine_tuned')

In [ ]:
def chat(prompt):
    if not prompt.strip():
        return "Please enter a prompt."
    
    # Tokenize input
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128
    ).to('cpu')
    
    # Make prediction
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        response = outputs.logits.item()
    
    return response

In [ ]:
# Add gradio Chatbot interface

demo = gr.ChatInterface(chat)

demo.launch()